### monitoring performace of a machine
support monitoring the remote machine or the VM in the remote machine 

* in file `sshConfig` to setup remote machine and vm credentials
* modify variable `monitoring_VM` to choose vm monitoring or not
* modify variable `INTERVAL` and `DURATION`
* set `True` or `False` for the monitoring items
* if monitoring network, need to modify `NET_INTERFACE` to set the net name
* if monitoring disk, need to modify `DISK_DEVICE` to set the disk name


In [ ]:
# 增强版：实时监控远程机器多项性能指标
monitoring_VM = True  # True 监控 VM 内部，False 监控跳板机
#monitoring_VM = False

INTERVAL = 5          # 采样间隔（秒）
DURATION = 60        # 总时长（秒），设为 None 则无限运行

# 开关：选择要监控的指标
MONITOR_CPU = True
MONITOR_MEM = True
MONITOR_DISK_USAGE = True
MONITOR_DISK_IO = False      # 需要 iostat（sysstat）
MONITOR_NETWORK = True
MONITOR_LOAD = True
MONITOR_PROCESSES = True
# 网络接口名称（通过 ip a 或 ifconfig 查看）
NET_INTERFACE = 'eth0'       # 例如 eth0, ens33
# 磁盘设备（用于 I/O 监控，如 sda）
DISK_DEVICE = 'sda'



In [ ]:
import paramiko
import time
import matplotlib.pyplot as plt
from IPython.display import clear_output
import os
from proxyJump import SSHViaJump
from proxyJump import SSHDirect
from proxyJump import parse_ssh_config

ssh_config_path = "sshConfig"
ssh_config = parse_ssh_config(ssh_config_path)
remote_config = ssh_config.get('remote', {})
if monitoring_VM:
    vm_config = ssh_config.get('vm', {})
    client = SSHViaJump(
        jump_host = remote_config.get('HostName'),
        jump_user = remote_config.get('User'),
        jump_key_path = remote_config.get('IdentityFile'),

        remote_host = vm_config.get('HostName'),
        remote_user = vm_config.get('User'),
        remote_key_path_on_jump = vm_config.get('IdentityFile'),
    )
else:
    client = SSHDirect(
        host = remote_config.get('HostName'),
        user = remote_config.get('User'),
        key_path = remote_config.get('IdentityFile'),
    )

# ==================== 数据存储 ====================
times = []
data = {
    'cpu': [], 'mem': [], 'disk_usage': [], 'disk_io': [],
    'net_sent': [], 'net_recv': [], 'load1': [], 'load5': [], 'load15': [],
    'processes': []
}
prev_net = None
start_time = time.time()

# ==================== 实时监控循环 ====================
print(f"Start monitoring {remote_config.get('HostName')}, refresh every {INTERVAL} seconds...")
print("Press the ■ button in the Jupyter toolbar to stop.\n")

while True:
    elapsed = time.time() - start_time
    if DURATION and elapsed >= DURATION:
        print("\nMonitoring duration reached, stopping...")
        break

    times.append(elapsed)

    try:
        # ---------- 采集各项指标 ----------
        # CPU
        if MONITOR_CPU:
            stdin, stdout, stderr = client.run(
                "top -b -n1 | grep '%Cpu' | awk '{print $2 + $4}'"
            )
            cpu_line = stdout.strip()
            data['cpu'].append(float(cpu_line.replace(',', '.')) if cpu_line else 0.0)

        # 内存
        if MONITOR_MEM:
            stdin, stdout, stderr = client.run(
                "free | grep Mem | awk '{print ($3/$2) * 100}'"
            )
            mem_line = stdout.strip()
            data['mem'].append(float(mem_line.replace(',', '.')) if mem_line else 0.0)

        # 磁盘使用率
        if MONITOR_DISK_USAGE:
            stdin, stdout, stderr = client.run(
                "df -h / | awk 'NR==2 {print $5}' | tr -d '%'"
            )
            disk_line = stdout.strip()
            data['disk_usage'].append(float(disk_line) if disk_line else 0.0)

        # 磁盘 I/O
        if MONITOR_DISK_IO:
            cmd = f"iostat -d -x 1 2 | grep {DISK_DEVICE} | tail -1 | awk '{{print $NF}}'"
            stdin, stdout, stderr = client.run(cmd)
            io_line = stdout.strip()
            data['disk_io'].append(float(io_line) if io_line else 0.0)

        # 网络
        if MONITOR_NETWORK:
            cmd = f"cat /proc/net/dev | grep {NET_INTERFACE} | awk '{{print $2, $10}}'"
            stdin, stdout, stderr = client.run(cmd)
            net_line = stdout.strip()
            if net_line:
                recv_bytes, sent_bytes = map(int, net_line.split())
                if prev_net is not None:
                    recv_rate = (recv_bytes - prev_net[0]) / INTERVAL
                    sent_rate = (sent_bytes - prev_net[1]) / INTERVAL
                    data['net_recv'].append(recv_rate)
                    data['net_sent'].append(sent_rate)
                else:
                    data['net_recv'].append(0.0)
                    data['net_sent'].append(0.0)
                prev_net = (recv_bytes, sent_bytes)
            else:
                data['net_recv'].append(0.0)
                data['net_sent'].append(0.0)

        # 平均负载
        if MONITOR_LOAD:
            stdin, stdout, stderr = client.run(
                "uptime | awk -F'load average:' '{print $2}' | awk '{print $1,$2,$3}'"
            )
            load_line = stdout.strip()
            if load_line:
                loads = load_line.replace(',', ' ').split()
                if len(loads) >= 3:
                    data['load1'].append(float(loads[0]))
                    data['load5'].append(float(loads[1]))
                    data['load15'].append(float(loads[2]))
                else:
                    data['load1'].append(0.0)
                    data['load5'].append(0.0)
                    data['load15'].append(0.0)
            else:
                data['load1'].append(0.0)
                data['load5'].append(0.0)
                data['load15'].append(0.0)

        # 进程数
        if MONITOR_PROCESSES:
            stdin, stdout, stderr = client.run("ps aux | wc -l")
            proc_line = stdout.strip()
            data['processes'].append(int(proc_line) if proc_line else 0)

    except Exception as e:
        print(f"采集失败: {e}")
        break

    # ---------- 绘图部分（与之前相同） ----------
    clear_output(wait=True)

    n_plots = 0
    if MONITOR_CPU: n_plots += 1
    if MONITOR_MEM: n_plots += 1
    if MONITOR_DISK_USAGE: n_plots += 1
    if MONITOR_DISK_IO: n_plots += 1
    if MONITOR_NETWORK: n_plots += 1
    if MONITOR_LOAD: n_plots += 1
    if MONITOR_PROCESSES: n_plots += 1

    if n_plots == 0:
        print("No metrics selected for monitoring. Please enable at least one metric.")
        break

    fig = plt.figure(figsize=(12, 2 * n_plots))
    gs = fig.add_gridspec(n_plots, 1, hspace=0.4)
    plot_idx = 0

    if MONITOR_CPU:
        ax = fig.add_subplot(gs[plot_idx])
        ax.plot(times, data['cpu'], 'r-', linewidth=1.5)
        ax.set_ylabel('CPU (%)')
        ax.set_ylim(0, 100)
        ax.grid(True)
        ax.set_title('CPU Usage')
        plot_idx += 1

    if MONITOR_MEM:
        ax = fig.add_subplot(gs[plot_idx])
        ax.plot(times, data['mem'], 'g-', linewidth=1.5)
        ax.set_ylabel('Memory (%)')
        ax.set_ylim(0, 100)
        ax.grid(True)
        ax.set_title('Memory Usage')
        plot_idx += 1

    if MONITOR_DISK_USAGE:
        ax = fig.add_subplot(gs[plot_idx])
        ax.plot(times, data['disk_usage'], 'b-', linewidth=1.5)
        ax.set_ylabel('Disk Usage (%)')
        ax.set_ylim(0, 100)
        ax.grid(True)
        ax.set_title('Root Disk Usage')
        plot_idx += 1

    if MONITOR_DISK_IO:
        ax = fig.add_subplot(gs[plot_idx])
        ax.plot(times, data['disk_io'], 'orange', linewidth=1.5)
        ax.set_ylabel('Disk I/O %util')
        ax.set_ylim(0, 100)
        ax.grid(True)
        ax.set_title(f'Disk I/O ({DISK_DEVICE})')
        plot_idx += 1

    if MONITOR_NETWORK and len(data['net_sent']) > 0:
        ax = fig.add_subplot(gs[plot_idx])
        ax.plot(times, data['net_sent'], 'c-', label='Sent', linewidth=1.5)
        ax.plot(times, data['net_recv'], 'm-', label='Received', linewidth=1.5)
        ax.set_ylabel('Network (bytes/s)')
        ax.legend()
        ax.grid(True)
        ax.set_title(f'Network Traffic ({NET_INTERFACE})')
        plot_idx += 1

    if MONITOR_LOAD:
        ax = fig.add_subplot(gs[plot_idx])
        ax.plot(times, data['load1'], label='1 min', linewidth=1.5)
        ax.plot(times, data['load5'], label='5 min', linewidth=1.5)
        ax.plot(times, data['load15'], label='15 min', linewidth=1.5)
        ax.set_ylabel('Load Average')
        ax.legend()
        ax.grid(True)
        ax.set_title('System Load')
        plot_idx += 1

    if MONITOR_PROCESSES:
        ax = fig.add_subplot(gs[plot_idx])
        ax.plot(times, data['processes'], 'purple', linewidth=1.5)
        ax.set_ylabel('Process Count')
        ax.grid(True)
        ax.set_title('Total Processes')
        plot_idx += 1

    fig.supxlabel('Time (seconds)')
    fig.suptitle(f'Remote Server Performance Monitor', fontsize=14)
    plt.tight_layout()
    plt.show()

    time.sleep(INTERVAL)

client.close()
print("Monitoring finished, SSH connection closed.")